In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import os

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score  #media armonica tra precisione e recall
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, losses
from tensorflow.keras.models import Model
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"Librerie caricate con successo. TensorFlow versione: {tf.__version__}")

In [ ]:
URL_DATASET = 'http://googleapis.com'
dataframe = pd.read_csv(URL_DATASET, header=None)

print(f"Dimensioni del dataset: {dataframe.shape}")
print(f"Distribuzione delle classi:\n{dataframe[140].value_counts(normalize=True)}") # L'ultima colonna è l'indice 140

raw_data = dataframe.values

# --- 2. Separazione Feature e Target ---
labels = raw_data[:, -1]
data = raw_data[:, 0:-1]

# --- 3. Divisione in Train e Test con STRATIFICAZIONE ---
# Usiamo il SEED = 42 definito nella prima cella per consistenza
train_data, test_data, train_labels, test_labels = train_test_split(
    data, 
    labels, 
    test_size=0.2, 
    random_state=SEED,  
    stratify=labels     # Assicura proporzioni identiche di anomalie nei due set
)

# --- 4. Normalizzazione MinMax (Cruciale per gli Autoencoder) ---
# Calcoliamo minimo e massimo SOLO sul train set
min_val = tf.reduce_min(train_data)
max_val = tf.reduce_max(train_data)

# Scaliamo entrambi i set tra 0 e 1
train_data = tf.cast((train_data - min_val) / (max_val - min_val), tf.float32)
test_data = tf.cast((test_data - min_val) / (max_val - min_val), tf.float32)

# Convertiamo i labes in booleani per comodità (True = Normale, False = Anomalia nel dataset TF)
train_labels = train_labels.astype(bool)
test_labels = test_labels.astype(bool)

print(f"\nDataset splittato e normalizzato:")
print(f"Train data shape: {train_data.shape}")
print(f"Test data shape: {test_data.shape}")


In [ ]:
train_data = tf.cast(train_data, tf.float32)
test_data = tf.cast(test_data, tf.float32)
min_val = tf.reduce_min(train_data)
max_val = tf.reduce_max(train_data)
train_data = (train_data - min_val) / (max_val - min_val)
test_data = (test_data - min_val) / (max_val - min_val)
print(f"I dati sono stati normalizzati con successo.")
print(f"Tipo dati finale: {train_data.dtype} | Min: {tf.reduce_min(train_data):.1f} | Max: {tf.reduce_max(train_data):.1f}")

In [ ]:
# Convertiamo i labels in array booleani di NumPy (corretto per creare le maschere)
train_labels = train_labels.astype(bool)
test_labels = test_labels.astype(bool)

# Usiamo tf.boolean_mask per un filtraggio nativo ed efficiente in TensorFlow
normal_train_data = tf.boolean_mask(train_data, train_labels)
normal_test_data = tf.boolean_mask(test_data, test_labels)

anomalous_train_data = tf.boolean_mask(train_data, ~train_labels)
anomalous_test_data = tf.boolean_mask(test_data, ~test_labels)

print("Separazione completata con successo:")
print(f"Campioni di Train -> Normali: {normal_train_data.shape[0]} | Anomalie: {anomalous_train_data.shape[0]}")
print(f"Campioni di Test  -> Normali: {normal_test_data.shape[0]}  | Anomalie: {anomalous_test_data.shape[0]}")

In [ ]:
num_samples = normal_train_data.shape[1]

plt.figure(figsize=(10, 4)) # Rende il grafico leggermente più largo e leggibile
plt.plot(np.arange(num_samples), normal_train_data[0].numpy(), color='#1f77b4', linewidth=1.5)

plt.title("A Normal ECG Signal", fontsize=12, fontweight='bold')
plt.xlabel("Time Steps (Samples)", fontsize=10)
plt.ylabel("Normalized Amplitude", fontsize=10)
plt.grid(True, linestyle='--', alpha=0.6) # Griglia più leggera e moderna

plt.tight_layout() # Ottimizza gli spazi del grafico
plt.show()

In [ ]:
num_samples = anomalous_train_data.shape[1]

plt.figure(figsize=(10, 4))
# Usiamo il rosso per differenziare visivamente l'anomalia rispetto al battito normale
plt.plot(np.arange(num_samples), anomalous_train_data[0].numpy(), color='#d62728', linewidth=1.5)

plt.title("An Anomalous ECG Signal", fontsize=12, fontweight='bold')
plt.xlabel("Time Steps (Samples)", fontsize=10)
plt.ylabel("Normalized Amplitude", fontsize=10)
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
class AnomalyDetector(Model):
    def __init__(self, input_dim):
        """
        Inizializza l'Autoencoder per l'Anomaly Detection.
        input_dim: int, il numero di feature/punti del segnale ECG (es. 140)
        """
        super().__init__() # Sintassi Python 3 standard e pulita
        
        # Encoder: comprime il segnale nello spazio latente (dimensione 8)
        self.encoder = tf.keras.Sequential([
            layers.Input(shape=(input_dim,)), # Definisce l'input esplicito per permettere il .summary()
            layers.Dense(32, activation="relu"),
            layers.Dense(16, activation="relu"),
            layers.Dense(8, activation="relu")
        ], name="Encoder")
        
        # Decoder: ricostruisce il segnale originale partendo dallo spazio latente
        self.decoder = tf.keras.Sequential([
            layers.Dense(16, activation="relu"),
            layers.Dense(32, activation="relu"),
            layers.Dense(input_dim, activation="sigmoid") # Dimensione dinamica, non più bloccata a 140
        ], name="Decoder")
        
    def call(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# Ricaviamo la dimensione dell'input dinamicamente dalle feature dei nostri dati
input_dimension = normal_train_data.shape[1]

# Istanziamo il modello passando la dimensione corretta
autoencoder = AnomalyDetector(input_dim=input_dimension)

# Avendo aggiunto layers.Input, adesso possiamo stampare il riepilogo del modello immediatamente!
autoencoder.build(input_shape=(None, input_dimension))
print("Architettura dell'Encoder:")
autoencoder.encoder.summary()
print("\nArchitettura del Decoder:")
autoencoder.decoder.summary()

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

autoencoder.compile(
    optimizer=optimizer, 
    loss=losses.MeanAbsoluteError(), # Usiamo l'oggetto classe nativo invece della stringa 'mae'
    metrics=['mse']                  # Monitoriamo anche il Mean Squared Error per avere più contesto
)


In [ ]:
history = autoencoder.fit(
    normal_train_data, 
    normal_train_data, # L'input e l'output desiderato sono identici negli autoencoder
    epochs=20, 
    batch_size=512,
    validation_data=(test_data, test_data),
    shuffle=True
)


In [ ]:
# 1. Applichiamo predict specificando un batch_size per evitare sovraccarichi di memoria
reconstructions = autoencoder.predict(normal_train_data, batch_size=32, verbose=0)

# 2. Calcoliamo il MAE convertendo esplicitamente in NumPy per consistenza di calcolo
# Usiamo losses.mae (o losses.MAE) assicurandoci che i tipi siano allineati
train_loss = tf.keras.losses.mae(reconstructions, normal_train_data.numpy()).numpy()

print(f"Calcolo dell'errore di ricostruzione completato.")
print(f"Forma del vettore train_loss: {train_loss.shape} (un valore di errore per ogni ECG)")
print(f"Errore MAE Medio sui dati normali: {train_loss.mean():.4f}")

In [ ]:
threshold = np.mean(train_loss) + np.std(train_loss)
print("Threshold: ", threshold)

In [ ]:
def predict(model, data, threshold, batch_size=32):
    """
    Classifica i dati ECG. Restituisce True se l'ECG è Normale (loss < threshold),
    altrimenti False (Anomalia).
    """
    # Usiamo predict con batch_size per non saturare la memoria ed essere più veloci
    reconstructions = model.predict(data, batch_size=batch_size, verbose=0)
    
    # Calcoliamo la loss allineando i tipi di dato (NumPy per stabilità)
    loss = tf.keras.losses.mae(reconstructions, tf.cast(data, tf.float32)).numpy()
    
    # Se la loss è MINORE della soglia, il battito è normale (True / 1)
    return loss < threshold

def print_stats(predictions, labels):
    """
    Calcola e stampa in modo chiaro tutte le metriche di classificazione,
    incluso l'F1-Score fondamentale per i dataset sbilanciati.
    """
    # Corretto il bug: ora usiamo la variabile 'predictions' passata come argomento della funzione
    acc = accuracy_score(labels, predictions)
    prec = precision_score(labels, predictions)
    rec = recall_score(labels, predictions)
    f1 = f1_score(labels, predictions) # Metrica aggiunta!
    
    print("-" * 30)
    print(f" PERFORMANCE DEL MODELLO")
    print("-" * 30)
    print(f"Accuracy  = {acc:.4f}")
    print(f"Precision = {prec:.4f}")
    print(f"Recall    = {rec:.4f}")
    print(f"F1-Score  = {f1:.4f}  <-- (Metrica chiave)")
    print("-" * 30)

In [ ]:
# CELLA FINALE1. Generiamo le predizioni sul test set usando la nostra funzione predict ottimizzata
preds = predict(autoencoder, test_data, threshold)

# 2. Stampiamo i risultati finali completi di F1-Score
print_stats(preds, test_labels)

In [ ]:
import time

def monitor_astronaut_live(single_raw_ecg, autoencoder, min_val, max_val, threshold):
    """
    Simula il sistema di monitoraggio a bordo.
    Riceve un singolo ECG grezzo, lo normalizza in sicurezza e lo valuta.
    """
    # 1. Pre-processing istantaneo e isolato (Float32)
    ecg_tensor = tf.cast(single_raw_ecg, tf.float32)  #cast in float 32 a partire da 64? bit
    
    # 2. Normalizzazione locale usando i parametri storici di calibrazione
    # Nota: NON ricalcoliamo min/max sul singolo battito, usiamo quelli storici del train!
    normalized_ecg = (ecg_tensor - min_val) / (max_val - min_val)
    
    # 3. TensorFlow richiede un batch. Trasformiamo il vettore singolo (140,) in un batch di un elemento (1, 140) compatibile con Tensor Flow
    input_batch = tf.expand_dims(normalized_ecg, axis=0) # matrice batch
    
    # 4. Ricostruzione (usiamo il modello direttamente per massima velocità sul singolo record)
    reconstruction = autoencoder(input_batch, training=False)  #disattivazione addestramento
    
    # 5. Calcolo dell'errore (MAE) per questo specifico battito
    mae_score = tf.reduce_mean(tf.abs(reconstruction - input_batch)).numpy() #media di sottrazione tra ECG ricostruito e quello originale tradotto in numpy
    
    # 6. Logica di Allarme
    is_anomaly = mae_score >= threshold  # if erro>= threshold ALLARM IS ON
    
    return is_anomaly, mae_score

# =====================================================================
# SIMULAZIONE DELLO STREAMING IN TEMPO REALE
# =====================================================================
print("📡 Avvio Monitoraggio Biometrico Astronauta (Simulazione Streaming)...")
print(f"Soglia critica impostata: {threshold:.4f}\n")

# Prendiamo 5 battiti normali e 5 anomali dal test set per simulare il flusso del sensore
test_stream_data = np.concatenate([anomalous_test_data.numpy()[:5], normal_test_data.numpy()[:5]])
test_stream_labels = np.concatenate([[False]*5, [True]*5]) # False = Anomalia, True = Normale

for i, raw_ecg in enumerate(test_stream_data):
    # Simula la frequenza cardiaca (es. un battito ogni 0.8 secondi)
    time.sleep(0.8) 
    
    # Esegui il controllo live
    alarm_triggered, score = monitor_astronaut_live(raw_ecg, autoencoder, min_val, max_val, threshold)
    
    # Stato reale (solo per verifica nel nostro test)
    real_status = "NORMALE" if test_stream_labels[i] else "⚠️ ANOMALIA REALE"
    
    # Output del Monitor di Bordo
    timestamp = time.strftime('%H:%M:%S')
    if alarm_triggered:
        print(f"[{timestamp}] [TELEMETRIA BATTITO {i+1}] MAE: {score:.4f} -> 🔴 CRITICAL ALERT: Rilevata Anomalia Cardiaca! (Stato Reale: {real_status})")
    else:
        print(f"[{timestamp}] [TELEMETRIA BATTITO {i+1}] MAE: {score:.4f} -> 🟢 Stato Nominale (Stato Reale: {real_status})")

print("\n📡 Simulazione terminata.")

#rimozione di fattore 140 per dinamismo del sistema 